[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/04_mpc.ipynb)

# Part 4 — MPC-style planning

> **Control flow: a simulator.** The LLM proposes candidate plans; a forward model ranks them.

Model Predictive Control, borrowed from process control: at each step, look ahead, plan the next
*N* actions against a world model, **execute only the first**, then re-plan from the new state.

The agentic version swaps one box — the LLM proposes the candidates, and the simulator (not the
LLM) decides which is best:

```
state ──▶ plan (LLM, N candidates) ──▶ simulate (forward model) ──▶ act (first step only)
  ▲                                                                          │
  └──────────────────────── re-plan from the new state ◀────────────────────┘
```

The LLM contributes what it is good at: proposing plausible candidates from context.
The simulator contributes what it is good at: **being right**.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json, re
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0"

from google import genai
from google.genai import types as gtypes
import numpy as np
from scipy.integrate import odeint

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)


def llm_text(system_prompt, user_prompt, temperature=0.2):
    resp = generate_with_retry(
        contents=user_prompt,
        config=gtypes.GenerateContentConfig(
            system_instruction=system_prompt, temperature=temperature))
    return resp.text or ""

print(f"Gemini client ready (model={MODEL}).")

### 4.1 The environment

A projectile with quadratic drag. The agent will only ever see `evaluate(angle) -> range`; it
does not get to look at the ODE.

$$\ddot{x} = -c_d |v|\dot{x}, \qquad \ddot{y} = -g - c_d|v|\dot{y}$$

with $v_0 = 50$ m/s, $c_d = 0.01$ m$^{-1}$, $g = 9.81$ m/s².

Without drag the optimum is exactly 45°. With drag it shifts lower — so the model's prior is
*almost* right, which is the interesting case.

In [ ]:
V0, CD, G = 50.0, 0.01, 9.81

def _rhs_drag(s, t):
    x, y, vx, vy = s
    v = np.sqrt(vx*vx + vy*vy)
    return [vx, vy, -CD*v*vx, -G - CD*v*vy]

def simulate_range(angle_deg):
    """Horizontal range (m) for a launch angle. This is the expensive action."""
    a = np.deg2rad(angle_deg)
    s0 = [0.0, 0.0, V0*np.cos(a), V0*np.sin(a)]
    t = np.linspace(0, 20, 4001)
    sol = odeint(_rhs_drag, s0, t)
    y = sol[:, 1]
    hit = np.where((y[:-1] >= 0) & (y[1:] < 0))[0]
    if len(hit) == 0:
        return float('nan')
    i = hit[0]
    frac = y[i] / (y[i] - y[i+1])
    return float(sol[i, 0] + frac * (sol[i+1, 0] - sol[i, 0]))

for a in (20.0, 45.0, 70.0):
    print(f"  {a:4.1f} deg -> {simulate_range(a):7.3f} m")

### 4.2 First, why the naive version is degenerate

An honest detour, because it is the mistake everyone makes on their first MPC agent.

MPC needs a forward model that is **cheaper than acting**. If you "simulate" a candidate angle by
calling `simulate_range` on it, you have already paid the full cost of the action — so
lookahead buys you nothing. You have built an expensive way to do a grid search.

**The fix:** the forward model has to be a genuine *surrogate*. Here, a quadratic least-squares
fit through the best few observations — essentially free, and accurate near the peak, which is
the only place accuracy matters.

In [ ]:
def fit_surrogate(obs, k=5):
    """Quadratic fit through the k best observations. This is the cheap forward model."""
    if len(obs) < 3:
        return None
    best = sorted(obs, key=lambda o: -o[1])[:max(k, 3)]
    a = np.array([o[0] for o in best], float)
    r = np.array([o[1] for o in best], float)
    if len(np.unique(a)) < 3:
        return None
    return np.polyfit(a, r, 2)

def surrogate_predict(coef, angle):
    return float(np.polyval(coef, angle))

# Three points are enough to locate the peak roughly.
demo = [(a, simulate_range(a)) for a in (20.0, 45.0, 70.0)]
coef = fit_surrogate(demo)
A, B, _ = coef
print(f"surrogate peak at {-B / (2 * A):.2f} deg")

### 4.3 The MPC loop

Read the loop for what the LLM is **not** allowed to do: it never decides which plan is best.
It generates hypotheses; the forward model adjudicates. That division is the entire pattern.

In [ ]:
def mpc_optimize(propose, evaluate, budget=10, n_candidates=5,
                 seeds=(20.0, 45.0, 70.0)):
    """Plan -> simulate (surrogate) -> act (ONE real evaluation) -> re-plan."""
    obs = [(a, evaluate(a)) for a in seeds]
    while len(obs) < budget:
        coef = fit_surrogate(obs)                                # 1. world model
        cands = [c for c in propose(obs, n_candidates) if 0 < c < 90]     # 2. plan
        if not cands:
            break
        # fit_surrogate returns None until there are 3 distinct angles
        if coef is None:
            best = cands[0]
        else:
            best = max(cands, key=lambda a: surrogate_predict(coef, a))   # 3. simulate
        obs.append((best, evaluate(best)))                       # 4. act: one only
        print(f"  n={len(obs):2d}  acted on {best:6.2f} -> {obs[-1][1]:7.3f}")
    return obs

In [ ]:
def llm_proposer(obs, n):
    """The LLM proposes candidates. It does NOT get to pick the winner."""
    hist = ", ".join(f"({a:.2f} -> {r:.2f})" for a, r in sorted(obs))
    txt = llm_text(
        "You propose candidate parameter values for an optimizer. "
        "Reply with ONLY a JSON list of numbers. No prose.",
        f"Observed (launch_angle_deg -> range_m): {hist}\n\n"
        f"Propose {n} new angles strictly between 0 and 90 worth testing next to "
        f"maximize range. Reply with only a JSON list.",
        temperature=0.7)
    return [float(m.group()) for m in re.finditer(r"\d+(?:\.\d+)?", txt)][:n]


obs = mpc_optimize(llm_proposer, simulate_range)
best_angle, best_range = max(obs, key=lambda o: o[1])
print(f"\nbest: {best_angle:.2f} deg -> {best_range:.3f} m in {len(obs)} evaluations")

### 4.4 When to reach for MPC

**Use it when:**
- a wrong action is **expensive or irreversible** — burning budget, committing a mesh, running an experiment;
- you have a **cheap forward model you trust more than the LLM**;
- actions compose, so looking *N* steps ahead genuinely differs from looking one step ahead.

**Don't bother when:**
- simulating a plan costs about what executing it costs — then just execute and observe (§4.2);
- you have **no** forward model, in which case MPC degenerates into the LLM grading its own
  homework: slower than ReAct and equally wrong;
- the problem is small and convex. Use `scipy.optimize` and go home.

